<a href="https://colab.research.google.com/github/abhimanyu1502/flyrank1st-assignment/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
### My Rule and Its Reason Codes

"""The baseline assigns pages to one of five archetypes using a priority-ordered rule cascade. Each archetype has a reason code used to explain recommendations to the content team.

Rule cascade (evaluated top to bottom — first match wins):

1. Champion: impressions_90d >= 1000 AND avg_position <= 10 AND days_since_last_update < 180
   Reason code: CHAMPION — high-visibility, well-ranked, recently refreshed. Maintain momentum.

2. Stale Visible: impressions_90d >= 500 AND avg_position <= 20 AND days_since_last_update >= 180
   Reason code: STALE_VISIBLE — strong rankings but content is aging. Schedule a refresh.

3. Hidden Gem: avg_position > 10 AND avg_position <= 30 AND ctr >= 0.02
   Reason code: HIDDEN_GEM — good click rate despite mid-tier position. Optimise title and meta.

4. Low Engagement: impressions_90d >= 250 AND engagement_rate < 30.0
   Reason code: LOW_ENGAGEMENT — visible but users are not staying. Review content relevance.

5. Weak / Low Demand: everything else (impressions_90d < 250)
   Reason code: WEAK_DEMAND — low organic reach. Deprioritise or consolidate."""

'The baseline assigns pages to one of five archetypes using a priority-ordered rule cascade. Each archetype has a reason code used to explain recommendations to the content team.\n\nRule cascade (evaluated top to bottom — first match wins):\n\n1. Champion: impressions_90d >= 1000 AND avg_position <= 10 AND days_since_last_update < 180\n   Reason code: CHAMPION — high-visibility, well-ranked, recently refreshed. Maintain momentum.\n\n2. Stale Visible: impressions_90d >= 500 AND avg_position <= 20 AND days_since_last_update >= 180\n   Reason code: STALE_VISIBLE — strong rankings but content is aging. Schedule a refresh.\n\n3. Hidden Gem: avg_position > 10 AND avg_position <= 30 AND ctr >= 0.02\n   Reason code: HIDDEN_GEM — good click rate despite mid-tier position. Optimise title and meta.\n\n4. Low Engagement: impressions_90d >= 250 AND engagement_rate < 30.0\n   Reason code: LOW_ENGAGEMENT — visible but users are not staying. Review content relevance.\n\n5. Weak / Low Demand: everythin

In [6]:
import pandas as pd
import numpy as np

In [7]:
# Load and filter dataset
df = pd.read_csv("/content/content_refresh_anonymized.csv")
df_clean = df[(df['impressions_90d'] >= 10) & (df['content_age_days'] >= 90)].copy()

def assign_baseline_archetype(row):
    if row['impressions_90d'] >= 1000 and row['avg_position'] <= 10 and row['days_since_last_update'] < 180:
        return 'Champion', 'CHAMPION'
    elif row['impressions_90d'] >= 500 and row['avg_position'] <= 20 and row['days_since_last_update'] >= 180:
        return 'Stale Visible', 'STALE_VISIBLE'
    elif row['avg_position'] > 10 and row['avg_position'] <= 30 and row['ctr'] >= 0.02:
        return 'Hidden Gem', 'HIDDEN_GEM'
    elif row['impressions_90d'] >= 250 and row['engagement_rate'] < 30.0:
        return 'Low Engagement', 'LOW_ENGAGEMENT'
    else:
        return 'Weak / Low Demand', 'WEAK_DEMAND'

df_clean[['baseline_archetype', 'reason_code']] = df_clean.apply(
    assign_baseline_archetype, axis=1, result_type='expand'
)

print("=== HEURISTIC BASELINE ARCHETYPE DISTRIBUTION ===")
print(df_clean['baseline_archetype'].value_counts(normalize=True).map('{:.1%}'.format))

ParserError: Error tokenizing data. C error: Expected 44 fields in line 4662, saw 66


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
### Build the Ranked Queue

"""We compute a composite action score for each page using three normalised signals:
- Log-scaled impression volume (weight 0.4) — prioritises high-reach pages
- Inverse position score (weight 0.35) — prioritises well-ranked pages
- Freshness penalty (weight 0.25) — surfaces pages that have not been updated recently

The score is normalised to [0, 1] and the full ranked queue is written to work/outputs/baseline_action_score.csv."""

'We compute a composite action score for each page using three normalised signals:\n- Log-scaled impression volume (weight 0.4) — prioritises high-reach pages\n- Inverse position score (weight 0.35) — prioritises well-ranked pages\n- Freshness penalty (weight 0.25) — surfaces pages that have not been updated recently\n\nThe score is normalised to [0, 1] and the full ranked queue is written to work/outputs/baseline_action_score.csv.'

In [3]:
import os

# Compute composite action score
df_clean['log_impressions'] = np.log1p(df_clean['impressions_90d'])
df_clean['position_score'] = 1 / (df_clean['avg_position'] + 1)
df_clean['staleness_score'] = df_clean['days_since_last_update'] / (df_clean['content_age_days'] + 1)

# Normalise each component to [0, 1]
def minmax(s):
    return (s - s.min()) / (s.max() - s.min() + 1e-9)

df_clean['score_impressions'] = minmax(df_clean['log_impressions'])
df_clean['score_position']    = minmax(df_clean['position_score'])
df_clean['score_staleness']   = minmax(df_clean['staleness_score'])

# Weighted composite
df_clean['action_score'] = (
    0.40 * df_clean['score_impressions'] +
    0.35 * df_clean['score_position'] +
    0.25 * df_clean['score_staleness']
)

# Sort and write output
df_ranked = df_clean.sort_values('action_score', ascending=False).reset_index(drop=True)
df_ranked['rank'] = df_ranked.index + 1

output_cols = ['rank', 'action_score', 'baseline_archetype', 'reason_code',
               'impressions_90d', 'avg_position', 'ctr', 'days_since_last_update', 'engagement_rate']

os.makedirs("../../work/outputs", exist_ok=True)
df_ranked[output_cols].to_csv("../../work/outputs/baseline_action_score.csv", index=False)

print(f"Ranked queue written: {len(df_ranked)} pages")
print(df_ranked[output_cols].head(5).to_string(index=False))

NameError: name 'np' is not defined

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.